In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "C:/Windows/Fonts/malgun.ttf"
font_prop = fm.FontProperties(fname=font_path)

plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
restaurants = [
    "담금", "익선취향", "빠오즈푸", "이쿠",
    "땀땀", "마복림할머니집", "다온카츠", "트라토리아 진"
]

edges = [
    ("담금", "익선취향", 252),
    ("담금", "트라토리아 진", 245),
    ("익선취향", "빠오즈푸", 65),
    ("빠오즈푸", "이쿠", 213),
    ("이쿠", "땀땀", 120),
    ("땀땀", "마복림할머니집", 139),
    ("빠오즈푸", "다온카츠", 280),
    ("트라토리아 진", "다온카츠", 249),
]

In [ ]:
G = nx.Graph()
G.add_nodes_from(restaurants)

for a, b, time in edges:
    G.add_edge(a, b, weight=time)

In [ ]:
plt.figure(figsize=(12, 8))

pos = nx.kamada_kawai_layout(G)

nx.draw(
    G, pos,
    with_labels=True,
    node_color="skyblue",
    node_size=3500,
    font_size=11,
    font_family=font_prop.get_name()
)

edge_labels = nx.get_edge_attributes(G, "weight")

nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels=edge_labels,
    font_size=10,
    font_family=font_prop.get_name()
)

plt.title("음식점 도보 이동 시간 그래프", fontsize=16, fontproperties=font_prop)
plt.axis("off")
plt.show()

In [ ]:
!pip install imageio imageio-ffmpeg

In [ ]:
import imageio.v2 as imageio
import numpy as np

def draw_graph_frame(title, visited_nodes=None, highlight_edges=None, filename=None):
    visited_nodes = visited_nodes or []
    highlight_edges = highlight_edges or []

    plt.figure(figsize=(12, 8))

    node_colors = [
        "orange" if node in visited_nodes else "skyblue"
        for node in G.nodes()
    ]

    edge_colors = []
    for u, v in G.edges():
        if (u, v) in highlight_edges or (v, u) in highlight_edges:
            edge_colors.append("red")
        else:
            edge_colors.append("gray")

    nx.draw(
        G, pos,
        with_labels=True,
        node_color=node_colors,
        edge_color=edge_colors,
        node_size=3500,
        font_size=11,
        font_family=font_prop.get_name(),
        width=2
    )

    edge_labels = nx.get_edge_attributes(G, "weight")
    nx.draw_networkx_edge_labels(
        G, pos,
        edge_labels=edge_labels,
        font_size=10,
        font_family=font_prop.get_name()
    )

    plt.title(title, fontsize=16, fontproperties=font_prop)
    plt.axis("off")

    if filename:
        plt.savefig(filename)
    plt.close()


def make_mp4(frame_files, output_name, fps=1):
    with imageio.get_writer(output_name, fps=fps) as writer:
        for file in frame_files:
            image = imageio.imread(file)
            writer.append_data(image)

In [ ]:
import os

os.makedirs("frames_dfs", exist_ok=True)

start = "빠오즈푸"
visited = []
dfs_edges = []
frame_files = []

def dfs(v):
    visited.append(v)
    frame_file = f"frames_dfs/frame_{len(frame_files):03d}.png"
    draw_graph_frame(f"DFS 방문: {v}", visited, dfs_edges, frame_file)
    frame_files.append(frame_file)

    for neighbor in G.neighbors(v):
        if neighbor not in visited:
            dfs_edges.append((v, neighbor))
            dfs(neighbor)

dfs(start)

make_mp4(frame_files, "dfs.mp4", fps=1)

print("dfs.mp4 저장 완료")

In [ ]:
from collections import deque
import os

os.makedirs("frames_bfs", exist_ok=True)

start = "빠오즈푸"
visited = [start]
queue = deque([start])
bfs_edges = []
frame_files = []

frame_file = f"frames_bfs/frame_000.png"
draw_graph_frame(f"BFS 시작: {start}", visited, bfs_edges, frame_file)
frame_files.append(frame_file)

while queue:
    v = queue.popleft()

    for neighbor in G.neighbors(v):
        if neighbor not in visited:
            visited.append(neighbor)
            queue.append(neighbor)
            bfs_edges.append((v, neighbor))

            frame_file = f"frames_bfs/frame_{len(frame_files):03d}.png"
            draw_graph_frame(f"BFS 방문: {neighbor}", visited, bfs_edges, frame_file)
            frame_files.append(frame_file)

make_mp4(frame_files, "bfs.mp4", fps=1)

print("bfs.mp4 저장 완료")

In [ ]:
import os

os.makedirs("frames_prim", exist_ok=True)

start = "빠오즈푸"
visited = {start}
prim_edges = []
frame_files = []

frame_file = "frames_prim/frame_000.png"
draw_graph_frame(f"Prim 시작: {start}", list(visited), prim_edges, frame_file)
frame_files.append(frame_file)

while len(visited) < len(G.nodes()):
    candidate_edges = []

    for u in visited:
        for v in G.neighbors(u):
            if v not in visited:
                candidate_edges.append((u, v, G[u][v]["weight"]))

    u, v, w = min(candidate_edges, key=lambda x: x[2])

    visited.add(v)
    prim_edges.append((u, v))

    frame_file = f"frames_prim/frame_{len(frame_files):03d}.png"
    draw_graph_frame(f"Prim 선택: {u} - {v} ({w}분)", list(visited), prim_edges, frame_file)
    frame_files.append(frame_file)

make_mp4(frame_files, "prim.mp4", fps=1)

print("prim.mp4 저장 완료")

In [ ]:
import os

os.makedirs("frames_kruskal", exist_ok=True)

parent = {}

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a
        return True
    return False

for node in G.nodes():
    parent[node] = node

sorted_edges = sorted(
    G.edges(data=True),
    key=lambda x: x[2]["weight"]
)

kruskal_edges = []
visited_nodes = []
frame_files = []

frame_file = "frames_kruskal/frame_000.png"
draw_graph_frame("Kruskal 시작", visited_nodes, kruskal_edges, frame_file)
frame_files.append(frame_file)

for u, v, data in sorted_edges:
    w = data["weight"]

    if union(u, v):
        kruskal_edges.append((u, v))

        if u not in visited_nodes:
            visited_nodes.append(u)
        if v not in visited_nodes:
            visited_nodes.append(v)

        frame_file = f"frames_kruskal/frame_{len(frame_files):03d}.png"
        draw_graph_frame(f"Kruskal 선택: {u} - {v} ({w}분)", visited_nodes, kruskal_edges, frame_file)
        frame_files.append(frame_file)

    if len(kruskal_edges) == len(G.nodes()) - 1:
        break

make_mp4(frame_files, "kruskal.mp4", fps=1)

print("kruskal.mp4 저장 완료")